In [1]:
import os
import json
import glob
import pandas as pd
import xlrd
import re
from pathlib import Path
import numpy as np
import xlsxwriter
import openpyxl
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter
from openpyxl.styles import Alignment, Border, Side, PatternFill

In [2]:
def apply_join(df_ML, df_state, df_samples, df_districts, df_crops):
    #df_ML left join df_state ON  'STATE'
    for feature in ['STATENAME', 'SHORTSTATE', 'HSTATENAME']:
        df_ML[feature] = (df_ML['STATE']
                .map(df_state.set_index('STATE')[feature])
                .fillna(df_ML[feature])
            )
    #df_ML left join df_samples ON  'SAMPLE'
    for feature in ['SAMPLENAME', 'HSAMPLENAME']:
        df_ML[feature] = (df_ML['SAMPLE']
                .map(df_samples.set_index('SAMPLE')[feature])
                .fillna(df_ML[feature])
            )
    #df_ML left join df_districts ON  'DISTRICT'
    for feature in ['DISTRICTNAME', 'HDISTRICTNAME', 'ROCODE', 'RONAME', 'HRONAME', 'SROCODE', 'SRONAME', 'HSRONAME']:
        df_ML[feature] = (df_ML['DISTRICT']
                .map(df_districts.set_index('DISTRICT')[feature])
                .fillna(df_ML[feature])
            )
    #df_ML left join df_crops ON  'CROPCODE'
    for feature in ['CROPNAME', 'HCROPNAME', 'SEASONNAME', 'HSEASONNAME']:
        df_ML[feature] = (df_ML['CROPCODE']
                .map(df_crops.set_index('CROPCODE')[feature])
                .fillna(df_ML[feature])
            )
    return df_ML

In [3]:
def get_first_row(worksheet, row_text_list):
    for text in row_text_list:
        text = text.lower()
        for row in worksheet.iter_rows():
            for cell in row:
                if cell.value and text in str(cell.value).lower():
                    return cell.row
    return -1

In [4]:
def get_first_col(worksheet, col_text_list):
    for text in col_text_list:
        text = text.lower()
        for row in worksheet.iter_rows():
            for cell in row:
                if cell.value and text in str(cell.value).lower():
                    return cell.column
    return -1

In [5]:
def correct_distt_name(incorrect_distt):
    with open("distt_correction.json", "r") as file:
        distt_correction = json.load(file)

    for correct, incorrect_list in distt_correction.items():
        for incorrect in incorrect_list:
            if incorrect.lower() in incorrect_distt.lower():
                correct_distt = correct
                return correct_distt

    return incorrect_distt

In [6]:
def apply_join(df_ML, df_state, df_samples, df_districts, df_seasons):
    #df_ML left join df_state ON  'STATE'
    for feature in ['STATENAME','SHORTSTATE']:
        df_ML[feature] = (df_ML['STATE']
                .map(df_state.set_index('STATE')[feature])
                .fillna(df_ML[feature])
            )
    #df_ML left join df_samples ON  'SAMPLE'
    for feature in ['SAMPLENAME']:
        df_ML[feature] = (df_ML['SAMPLE']
                .map(df_samples.set_index('SAMPLE')[feature])
                .fillna(df_ML[feature])
            )
    #df_ML left join df_districts ON  'DISTRICT'
    for feature in ['ROCODE','RONAME','SROCODE','SRONAME','DISTRICTNAME']:
        df_ML[feature] = (df_ML['DISTRICT']
                .map(df_districts.set_index('DISTRICT')[feature])
                .fillna(df_ML[feature])
            )
    #df_ML left join df_seasons ON  'SEASON'
    for feature in ['SEASONNAME']:
        df_ML[feature] = (df_ML['SEASONCODE']
                .map(df_seasons.set_index('SEASONCODE')[feature])
                .fillna(df_ML[feature])
            )
    return df_ML

In [7]:
def get_SL_to_ML(directory):
    #df to use
    df_state = pd.read_excel("ML_Template.xlsx", sheet_name="State", dtype=str)
    state = df_state['STATE'].iloc[0]
    year = df_state['YEAR'].iloc[0]

    df_seasons = pd.read_excel("ML_Template.xlsx", sheet_name="Seasons", dtype=str)
    seasons = df_seasons['SEASONCODE'].unique().tolist()

    
    df_samples = pd.read_excel("ML_Template.xlsx", sheet_name="Samples", dtype=str)
    df_districts = pd.read_excel("ML_Template.xlsx", sheet_name="Districts", dtype=str)
    

    #df to create
    df_ML = pd.DataFrame(columns=['YR','SAMPLE','SAMPLENAME','SEASONCODE','SEASONNAME',
                                  'STATE','STATENAME','SHORTSTATE','ROCODE',
                                  'RONAME','SROCODE','SRONAME','DISTRICT','DISTRICTNAME',
                                  'STRTM','SELORDER','THSL','VILL'])
    
    # Get all .xlsx and .xls files in the directory
    excel_files = glob.glob(os.path.join(directory, "*.xlsx"))
    
    for file_path in excel_files:
        file_name = os.path.basename(file_path)
        incorrect_distt = Path(file_name).stem.strip().title()

        district_name = correct_distt_name(incorrect_distt)
        district_code = df_districts[df_districts['DISTRICTNAME']==district_name]['DISTRICT'].iloc[0]
        # Determine file type and use appropriate library
        if file_path.endswith('.xlsx'):
            #try:
            wb = load_workbook(file_path, data_only=True)
            for sheet_name in wb.sheetnames:
                if "cs" in sheet_name.lower() or "ss" in sheet_name.lower():
                    sample = '1' if "cs" in sheet_name.lower() else '2'
                    
                    ws = wb[sheet_name]
                    
                    # setting which cells to scan
                    row_text_list = ['stratum', 'tkname', 'bkname', 'vgname', 'order', 'selection']
                    start_row = 0
                    title_row = 0
                    while start_row <= 0:
                        title_row = get_first_row(ws, row_text_list)
                        start_row = title_row + 1
                    end_row = ws.max_row + 1
                    
                    if start_row > 0:
                        for row_idx in range(start_row, end_row):
                            strtm = ws.cell(row=row_idx, column=1).value
                            if strtm is None:
                                strtm_row = row_idx
                                while strtm is None and strtm_row > title_row:
                                    strtm_row -= 1
                                    strtm = ws.cell(row=strtm_row, column=1).value
                            
                            taluka = ws.cell(row=row_idx, column=2).value
                            village = ws.cell(row=row_idx, column=4).value
                            selorder = ws.cell(row=row_idx, column=6).value

                            if strtm is not None and taluka is not None and village is not None and selorder is not None:
                                for season in seasons:
                                    df_ML_row = {
                                        'YR': year,
                                        'SAMPLE': sample,
                                        'SEASONCODE': season,
                                        'STATE': state,
                                        'DISTRICT': district_code,
                                        'STRTM': strtm,
                                        'SELORDER': selorder,
                                        'THSL': taluka,
                                        'VILL': village
                                    }
                                    df_ML = pd.concat([df_ML, pd.DataFrame([df_ML_row], dtype=str)], ignore_index=True)

            #except Exception as e:
            #    print(f"Error reading {file_name}, {sheet_name}, {row_idx}, {selorder}")
            #    raise e

    #joins
    df_ML = apply_join(df_ML, df_state, df_samples, df_districts, df_seasons)
    df_ML.sort_values(['SEASONCODE','SAMPLE','ROCODE','SROCODE','DISTRICT','STRTM','SELORDER'], inplace=True)
    return df_ML

In [8]:
if __name__ == "__main__":
    curr_dir = Path.cwd()
    directory = curr_dir / "SL 1.0"
    excel_path = curr_dir / "1.0_SL_to_ML.xlsx"
    df_ML = get_SL_to_ML(directory)
    
    with pd.ExcelWriter(excel_path, engine='xlsxwriter') as writer:
        df_ML.to_excel(writer, sheet_name="ML", index=False)